# Prepare latest 2026 rainfall

Imports, input/output paths, and the shared state-name mapping used by the dengue preparation notebook.


In [ ]:
import json

import pandas as pd

INPUT_FILE = "original/2026_district_wise_rain_data.csv"
OUTPUT_FILE = "data_used/rain_latest.json"

NAME_MAPPING = {
    # District spellings
    "pudukkottai": "pudukottai",
    # The rainfall file combines these two former states
    "dadra and nagar haveli": "the dadra and nagar haveli and daman and diu",
    "daman and diu": "the dadra and nagar haveli and daman and diu",
    # 2026 file spelling
    "andaman and nicobar": "andaman and nicobar islands",
}

print("Input file: ", INPUT_FILE)
print("Output file:", OUTPUT_FILE)


## 1. Name-cleaning helper

Trim whitespace, lowercase, replace `&` with `and`, collapse extra spaces, then apply the mapping.


In [ ]:
def clean_name(value):
    if pd.isna(value):
        return value
    name = str(value).strip().lower().replace("&", "and")
    name = " ".join(name.split())
    return NAME_MAPPING.get(name, name)

for example in ["ANDAMAN & NICOBAR", "Dadra & Nagar Haveli", "Tamil Nadu"]:
    print(example, "->", clean_name(example))


## 2. Load the rainfall file

Read the raw CSV, print every column name, and check the number of rows and the date range.


In [ ]:
rain = pd.read_csv(INPUT_FILE)

print("Columns:")
print(rain.columns.tolist())
print("Rows loaded:", len(rain))

date_column = next(
    column for column in rain.columns
    if str(column).strip().lower() == "date"
)
dates = pd.to_datetime(rain[date_column], errors="coerce")
print("Date min:", dates.min())
print("Date max:", dates.max())


## 3. Keep and rename the needed columns

Select only state, district, date, daily actual, and daily normal. Column names are matched case-insensitively, and all weekly/cumulative/monthly columns are ignored.


In [ ]:
needed_columns = {
    "state": "state",
    "district": "district",
    "date": "date",
    "actual": "daily actual",
    "normal": "daily normal",
}

selected_columns = {}
for new_name, wanted_name in needed_columns.items():
    match = next(
        (column for column in rain.columns
         if str(column).strip().lower() == wanted_name),
        None,
    )
    if match is None:
        raise KeyError(f"Missing column: {wanted_name}")
    selected_columns[new_name] = match

rain = rain[list(selected_columns.values())].rename(
    columns={source: new_name for new_name, source in selected_columns.items()}
)
rain["date"] = pd.to_datetime(rain["date"], errors="coerce")
rain["state"] = rain["state"].map(clean_name)

print("Selected columns:", selected_columns)
print("Rows with invalid dates:", rain["date"].isna().sum())
rain.head()


## 4. Clean values and duplicates

Convert rainfall values to numeric, drop rows missing either value, and keep the last row for each state-district-date combination.


In [ ]:
print("Starting rows:", len(rain))

rain["actual"] = pd.to_numeric(rain["actual"], errors="coerce")
rain["normal"] = pd.to_numeric(rain["normal"], errors="coerce")
before = len(rain)
rain = rain.dropna(subset=["actual", "normal"])
print("After dropping missing actual/normal:", len(rain), f"(dropped {before - len(rain)})")

before = len(rain)
rain = rain.drop_duplicates(subset=["state", "district", "date"], keep="last")
print("After removing duplicate state/district/date rows:", len(rain), f"(dropped {before - len(rain)})")

rain.head()


## 5. Build state-month totals

Create `YYYY-MM` months, sum actual and normal rainfall, and count distinct days and districts. Completeness compares days present with the number of days in the month.


In [ ]:
rain["month"] = rain["date"].dt.strftime("%Y-%m")

rain_monthly = (
    rain
    .groupby(["state", "month"], as_index=False)
    .agg(
        actual_sum=("actual", "sum"),
        normal_sum=("normal", "sum"),
        days_present=("date", "nunique"),
        districts=("district", "nunique"),
    )
)

rain_monthly["departure"] = (
    (rain_monthly["actual_sum"] - rain_monthly["normal_sum"])
    / rain_monthly["normal_sum"].where(rain_monthly["normal_sum"] != 0)
)
rain_monthly["days_in_month"] = pd.PeriodIndex(
    rain_monthly["month"], freq="M"
).days_in_month
rain_monthly["is_complete"] = (
    rain_monthly["days_present"] == rain_monthly["days_in_month"]
)

rain_monthly = rain_monthly.sort_values(["state", "month"]).reset_index(drop=True)
print("State-month rows:", len(rain_monthly))
rain_monthly


## 6. Check coverage

Print all incomplete state-months and the latest complete month for every state.


In [ ]:
incomplete = rain_monthly[~rain_monthly["is_complete"]]
print("Incomplete state-months:", len(incomplete))
print(
    incomplete[["state", "month", "days_present", "days_in_month", "districts"]]
    .to_string(index=False)
)

complete = rain_monthly[rain_monthly["is_complete"]]
latest_complete = (
    complete.groupby("state")["month"]
    .max()
    .reindex(sorted(rain_monthly["state"].unique()))
    .fillna("none")
)
print("\nLatest complete month per state:")
print(latest_complete.to_string())


## 7. Compare state names with the dengue monthly table

Read `processed/dengue_monthly.csv` only for a name check. Nothing is changed based on the result.


In [ ]:
try:
    dengue_monthly = pd.read_csv("processed/dengue_monthly.csv", usecols=["state"])
except FileNotFoundError:
    print("processed/dengue_monthly.csv not found; reading data_used/dengue_monthly.csv")
    dengue_monthly = pd.read_csv("data_used/dengue_monthly.csv", usecols=["state"])

dengue_states = set(dengue_monthly["state"].dropna().unique())
rain_states = set(rain_monthly["state"].dropna().unique())

print("In rain_latest but not in dengue_monthly:")
print(sorted(rain_states - dengue_states))
print("\nIn dengue_monthly but missing from rain_latest:")
print(sorted(dengue_states - rain_states))


## 8. Save the JSON table

Write one record per state-month, converting missing values to `null`.


In [ ]:
records = []
for row in rain_monthly.itertuples(index=False):
    records.append({
        "state": row.state,
        "month": row.month,
        "actual_sum": None if pd.isna(row.actual_sum) else float(row.actual_sum),
        "normal_sum": None if pd.isna(row.normal_sum) else float(row.normal_sum),
        "departure": None if pd.isna(row.departure) else float(row.departure),
        "days_present": int(row.days_present),
        "districts": int(row.districts),
        "is_complete": bool(row.is_complete),
    })

with open(OUTPUT_FILE, "w") as output_file:
    json.dump(records, output_file, indent=2)

print("Saved rows:", len(records))
print("Latest month present:", rain_monthly["month"].max())
print("Saved to:", OUTPUT_FILE)
